# 10 — Fetch DDF-RA `dataStructure.yml`

Pins the DDF-RA release tag and downloads
`Deliverables/UML/dataStructure.yml` into `../downloads/`.

Bumping the pinned tag is a deliberate action — see `CLAUDE.md`.


## Pinned configuration


In [ ]:
DDF_RA_TAG  = "v4.0.0"
DDF_RA_REPO = "cdisc-org/DDF-RA"
SOURCE_PATH = "Deliverables/UML/dataStructure.yml"
TARGET_PATH = "../downloads/dataStructure.yml"
META_PATH   = "../downloads/.fetch_meta.json"


## Build the raw URL


In [ ]:
RAW_URL = f"https://raw.githubusercontent.com/{DDF_RA_REPO}/{DDF_RA_TAG}/{SOURCE_PATH}"
print(RAW_URL)


## Fetch via `urllib` (standard library)


In [ ]:
import urllib.request
from pathlib import Path

Path(TARGET_PATH).parent.mkdir(parents=True, exist_ok=True)

req = urllib.request.Request(RAW_URL)
with urllib.request.urlopen(req) as resp:
    if resp.status != 200:
        raise RuntimeError(f"unexpected status {resp.status} for {RAW_URL}")
    body = resp.read()

Path(TARGET_PATH).write_bytes(body)
print(f"wrote {len(body)} bytes to {TARGET_PATH}")


## Confirm: size, header, SHA-256, sidecar metadata

Writes `downloads/.fetch_meta.json` for downstream notebooks to read
(tag, URL, SHA-256, size, fetch timestamp).


In [ ]:
import hashlib
import json
import datetime
from pathlib import Path

raw = Path(TARGET_PATH).read_bytes()
sha256 = hashlib.sha256(raw).hexdigest()
size = len(raw)

print(f"size:   {size} bytes")
print(f"sha256: {sha256}")
print()
print("first 10 lines:")
for line in raw.decode("utf-8").splitlines()[:10]:
    print(f"  {line}")

meta = {
    "ddf_ra_tag":     DDF_RA_TAG,
    "ddf_ra_repo":    DDF_RA_REPO,
    "source_path":    SOURCE_PATH,
    "raw_url":        RAW_URL,
    "sha256":         sha256,
    "size_bytes":     size,
    "fetched_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
Path(META_PATH).write_text(json.dumps(meta, indent=2) + "\n")
print(f"\nwrote sidecar metadata to {META_PATH}")


## Provenance

Fetched from `cdisc-org/DDF-RA@v4.0.0` at
`Deliverables/UML/dataStructure.yml`. Tag pinned in cell 1.


## v0.1 — Fetch USDM_CT.xlsx (same pinned tag)

`USDM_CT.xlsx` is the source of truth for codelist bindings on `Code`-typed
properties. It is co-pinned with `dataStructure.yml` at the same DDF-RA
release tag — bumping the tag refreshes both files in lockstep.


In [ ]:
CT_SOURCE_PATH = "Deliverables/CT/USDM_CT.xlsx"
CT_TARGET_PATH = "../downloads/USDM_CT.xlsx"
CT_META_PATH   = "../downloads/.fetch_meta_ct.json"

CT_RAW_URL = f"https://raw.githubusercontent.com/{DDF_RA_REPO}/{DDF_RA_TAG}/{CT_SOURCE_PATH}"
print(CT_RAW_URL)


In [ ]:
import urllib.request
from pathlib import Path

Path(CT_TARGET_PATH).parent.mkdir(parents=True, exist_ok=True)

req = urllib.request.Request(CT_RAW_URL)
with urllib.request.urlopen(req) as resp:
    if resp.status != 200:
        raise RuntimeError(f"unexpected status {resp.status} for {CT_RAW_URL}")
    body = resp.read()

Path(CT_TARGET_PATH).write_bytes(body)
print(f"wrote {len(body)} bytes to {CT_TARGET_PATH}")


In [ ]:
import hashlib
import json
import datetime
from pathlib import Path
import openpyxl

raw = Path(CT_TARGET_PATH).read_bytes()
sha256 = hashlib.sha256(raw).hexdigest()
size = len(raw)

wb = openpyxl.load_workbook(CT_TARGET_PATH, read_only=True, data_only=True)
sheet_names = wb.sheetnames
expected = ["DDF Entities&Attributes", "DDF valid value sets"]
if sheet_names != expected:
    raise RuntimeError(f"unexpected sheets in USDM_CT: {sheet_names!r}; expected {expected!r}")

print(f"size:        {size} bytes")
print(f"sha256:      {sha256}")
print(f"sheet_names: {sheet_names}")

meta = {
    "ddf_ra_tag":     DDF_RA_TAG,
    "ddf_ra_repo":    DDF_RA_REPO,
    "source_path":    CT_SOURCE_PATH,
    "raw_url":        CT_RAW_URL,
    "sha256":         sha256,
    "size_bytes":     size,
    "sheet_names":    sheet_names,
    "fetched_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
Path(CT_META_PATH).write_text(json.dumps(meta, indent=2) + "\n")
print(f"\nwrote sidecar metadata to {CT_META_PATH}")


## Provenance — USDM_CT

Fetched from `cdisc-org/DDF-RA@<tag>` at `Deliverables/CT/USDM_CT.xlsx`.
Tag is the same `DDF_RA_TAG` pinned in cell 1 — model and bindings are
always co-pinned.
